# 06 · Tóm tắt văn bản (Summarization)

Cách làm giống hệt dịch máy (notebook 05): model encoder–decoder, chỉ khác dữ liệu là **(văn bản dài → bản tóm tắt)**.

| Ngôn ngữ | Model gợi ý |
|---|---|
| Tiếng Việt | `VietAI/vit5-base` |
| Tiếng Anh | `facebook/bart-large-cnn` |
| Đa ngữ | `google/mt5-base` |

Lý thuyết: xem `knowledge/06_tom_tat.md`

In [ ]:
# !pip install -U transformers datasets accelerate sentencepiece

## 0. Import

In [ ]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. Cấu hình

In [ ]:
TRAIN_FILE = "train.csv"     # file của đề
VALID_FILE = "valid.csv"
TEST_FILE = None
TEXT_COL = "text"            # cột văn bản gốc
SUMMARY_COL = "summary"      # cột bản tóm tắt

MODEL_NAME = "VietAI/vit5-base"
MAX_INPUT_LEN = 1024
MAX_SUMMARY_LEN = 256

## 2. Đọc dữ liệu

In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def read_file(path):
    df = pd.read_csv(path).rename(columns={TEXT_COL: "text", SUMMARY_COL: "summary"})
    df = df.dropna(subset=["text", "summary"])
    df["text"] = df["text"].apply(normalize)
    df["summary"] = df["summary"].apply(normalize)
    return df.reset_index(drop=True)


train_df = read_file(TRAIN_FILE)
valid_df = read_file(VALID_FILE)
print(len(train_df), len(valid_df))

## 3. Chấm điểm ROUGE
Đếm số từ (ROUGE-1), cặp từ (ROUGE-2), chuỗi chung dài nhất (ROUGE-L) giữa bản tóm tắt máy và bản chuẩn.

In [ ]:
def f_score(overlap, pred_count, ref_count):
    if overlap == 0:
        return 0.0
    precision = overlap / pred_count
    recall = overlap / ref_count
    return 2 * precision * recall / (precision + recall)


def ngrams(words, n):
    return Counter(tuple(words[i:i + n]) for i in range(len(words) - n + 1))


def rouge_n(pred, ref, n):
    pred_ngrams = ngrams(pred.lower().split(), n)
    ref_ngrams = ngrams(ref.lower().split(), n)
    overlap = sum((pred_ngrams & ref_ngrams).values())
    return f_score(overlap, max(sum(pred_ngrams.values()), 1), max(sum(ref_ngrams.values()), 1))


def longest_common_subsequence(a, b):
    table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = table[i - 1][j - 1] + 1
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[-1][-1]


def rouge_l(pred, ref):
    pred_words = pred.lower().split()
    ref_words = ref.lower().split()
    lcs = longest_common_subsequence(pred_words, ref_words)
    return f_score(lcs, max(len(pred_words), 1), max(len(ref_words), 1))


def rouge(predictions, references):
    n = len(predictions)
    r1 = sum(rouge_n(p, r, 1) for p, r in zip(predictions, references)) / n
    r2 = sum(rouge_n(p, r, 2) for p, r in zip(predictions, references)) / n
    rl = sum(rouge_l(p, r) for p, r in zip(predictions, references)) / n
    return {"rouge1": round(100 * r1, 2), "rouge2": round(100 * r2, 2), "rougeL": round(100 * rl, 2)}

⚠️ Không dùng thư viện `rouge_score` mặc định cho tiếng Việt: nó xoá các ký tự có dấu.

## 4. Fine-tune

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)


def tokenize(batch):
    inputs = tokenizer(batch["text"], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch["summary"], max_length=MAX_SUMMARY_LEN, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs


train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True, remove_columns=["text", "summary"])
valid_ds = Dataset.from_pandas(valid_df).map(tokenize, batched=True, remove_columns=["text", "summary"])

In [ ]:
def compute_metrics(eval_pred):
    pred_ids, label_ids = eval_pred
    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)

    predictions = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    references = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return rouge(predictions, references)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="sum_output",
    learning_rate=3e-4,                  # T5/ViT5 cần lr cao hơn BERT
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # batch thật = 4 x 4 = 16
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    warmup_ratio=0.05,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    predict_with_generate=True,
    generation_max_length=MAX_SUMMARY_LEN,
    generation_num_beams=4,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),   # T5 không dùng fp16
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)
trainer.train()

## 5. Chọn cách sinh

In [ ]:
def summarize(texts, batch_size=8, **generate_options):
    model.eval()
    results = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LEN)
        inputs = inputs.to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=MAX_SUMMARY_LEN, **generate_options)
        results.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return results

In [ ]:
sample = valid_df.head(100)

options_to_try = [
    {"num_beams": 1},
    {"num_beams": 4},
    {"num_beams": 4, "no_repeat_ngram_size": 3},
    {"num_beams": 4, "no_repeat_ngram_size": 3, "length_penalty": 1.5},
]

for options in options_to_try:
    predictions = summarize(sample["text"].tolist(), **options)
    print(options, rouge(predictions, sample["summary"].tolist()))

## 6. Tóm tắt tập test và lưu bài nộp

In [ ]:
BEST_OPTIONS = {"num_beams": 4, "no_repeat_ngram_size": 3}   # sửa theo kết quả ô trên

if TEST_FILE is not None:
    test_df = pd.read_csv(TEST_FILE).rename(columns={TEXT_COL: "text"})
    test_df["text"] = test_df["text"].apply(normalize)
    test_df["summary"] = summarize(test_df["text"].tolist(), **BEST_OPTIONS)
    test_df.to_csv("submission.csv", index=False)